# Query Optimization Patterns

## Quick notes
### What this covers
- A catalogue of common slow-query patterns ("anti-patterns"), each with a rewrite that does less work.
- How to measure work without a clock: count SQLite virtual-machine steps and read the query plan.
- A repeatable workflow: reproduce, measure, read the plan, change one thing, measure again.
### Key points
- Optimize by **work done**, not by stopwatch. Steps and plan text are the same on every machine; seconds are not.
- Make the database do less: fewer rows read, fewer rows sorted, fewer round trips.
- Filter early, select only needed columns, let indexes do ordering, test existence with `EXISTS`.
- Rewrites must keep the **same result**. Always compare results before and after.
- Change one thing at a time, otherwise you do not know what helped.
### Gotchas (the part worth revising twice)
- `NOT IN` with a NULL in the list returns nothing at all. `NOT EXISTS` does not have this problem.
- `DISTINCT` that "fixes" duplicates usually hides a join that multiplies rows. Fix the join instead.
- `OFFSET 10000` reads and throws away 10000 rows. Keyset pagination ("after this key") does not.
- A function or arithmetic on an indexed column (`strftime(...)`, `amount * 2`) hides the index.
- Comparing a TEXT column to a number uses text rules: `'007'` is not `7`.
- A query that is fast on 5000 rows in a tutorial can still be a full scan; read the plan, do not guess.
- N+1 queries (one query per row from Python) are slow even if each query is fast.
### When it matters
- A query runs often, runs on a large table, or sits behind a web page.
- Never optimize a query that runs once on 100 rows. Measure first, then decide.
### One-line recap example
```sql
-- bad: function on the column, scans everything
SELECT * FROM orders WHERE strftime('%Y-%m', order_date) = '2024-03';
-- good: a range the index can use
SELECT order_id FROM orders WHERE order_date >= '2024-03-01' AND order_date < '2024-04-01';
```

## 01. Seed data: customers, orders, order items
- `customers`: 300 rows. Only customers 1 to 281 have orders, so 19 customers never ordered. Order status varies from customer to customer.
- `orders`: 5000 rows generated by a recursive CTE, plus one guest order with a NULL `customer_id` (used later for the `NOT IN` trap).
- `order_items`: 2 rows per order, 10000 rows.
- Everything is computed from the row number, so the data is identical on every run. Indexes come later, one at a time.
- See `07-Performance/01-Indexes-And-Query-Plans` for what indexes are; this notebook is about writing queries that use them well.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    country     TEXT NOT NULL,
    email       TEXT NOT NULL
);
CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER,
    status      TEXT NOT NULL,
    order_date  TEXT NOT NULL,
    amount      INTEGER NOT NULL,
    ref         TEXT NOT NULL
);
CREATE TABLE order_items (
    item_id  INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    sku      TEXT NOT NULL,
    qty      INTEGER NOT NULL
);

WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 300)
INSERT INTO customers
SELECT i,
       'Customer ' || printf('%03d', i),
       CASE i % 4 WHEN 0 THEN 'IN' WHEN 1 THEN 'US' WHEN 2 THEN 'DE' ELSE 'BR' END,
       'cust' || i || '@' || CASE i % 3 WHEN 0 THEN 'example.com' WHEN 1 THEN 'mail.org' ELSE 'corp.net' END
FROM n;

WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 5000)
INSERT INTO orders
SELECT i,
       1 + (i * 37) % 281,
       CASE i % 4 WHEN 0 THEN 'new' WHEN 1 THEN 'paid' WHEN 2 THEN 'shipped' ELSE 'returned' END,
       '2024-' || printf('%02d', 1 + (i % 12)) || '-' || printf('%02d', 1 + (i * 7) % 28),
       10 + (i * 13) % 490,
       printf('%05d', i)
FROM n;
INSERT INTO orders VALUES (5001, NULL, 'paid', '2024-06-15', 99, '05001');

WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 10000)
INSERT INTO order_items
SELECT i, 1 + (i - 1) / 2, 'SKU-' || printf('%02d', 1 + (i * 11) % 40), 1 + i % 5
FROM n;

SELECT (SELECT COUNT(*) FROM customers) AS customers,
       (SELECT COUNT(*) FROM orders) AS orders,
       (SELECT COUNT(*) FROM order_items) AS items;

customers | orders | items
----------+--------+------
300       | 5001   | 10000
(1 row)



## 02. Measuring work without a clock
- Clocks differ between machines and runs, so we count work instead.
- `con.set_progress_handler(fn, 1)` calls `fn` once per SQLite virtual-machine instruction. The number of calls is a **step count**: a fair "how much work" number that is the same every run.
- `EXPLAIN QUERY PLAN` shows the strategy (`SCAN` = read everything, `SEARCH` = jump via an index).
- Helper functions defined below are used in every later section: `steps(sql)`, `plan(sql)` and `compare({...})`.
- `compare` also checks that all variants return the **same rows**. A rewrite that changes the answer is a bug, not an optimization.

In [3]:
def steps(sql, params=(), every=1):
    """Run the query, return (virtual-machine steps, row count, rows).
    every=1 counts every instruction; a bigger value counts in coarser units (for very long queries)."""
    n = [0]
    def tick():
        n[0] += every
        return 0
    con.set_progress_handler(tick, every)
    try:
        rows = con.execute(sql, params).fetchall()
    finally:
        con.set_progress_handler(None, 1)
    return n[0], len(rows), rows

def plan(sql, params=()):
    """Print only the useful `detail` text of EXPLAIN QUERY PLAN, indented by nesting."""
    depth = {0: 0}
    for id_, parent, _, detail in con.execute("EXPLAIN QUERY PLAN " + sql, params):
        depth[id_] = depth.get(parent, 0) + 1
        print("  " * (depth[id_] - 1) + detail)

def compare(variants, params=(), check=True, every=1):
    """variants: {label: sql}. Print steps and rows per variant and whether results match."""
    results = {}
    for label, sql in variants.items():
        n, count, rows = steps(sql, params, every)
        results[label] = sorted(rows, key=repr)
        print(f"{label:<28} steps={n:>8,}  rows={count}")
    first = next(iter(results.values()))
    if check:
        print("all results identical:", all(r == first for r in results.values()))

print("helpers ready")

helpers ready


## 03. The optimization workflow
- **1. Reproduce**: get the exact slow query and realistic data. A fast query on tiny data proves nothing.
- **2. Measure**: record the baseline (steps here, wall time in real life) and the number of rows returned.
- **3. Plan**: read `EXPLAIN QUERY PLAN`. Find the `SCAN` of a big table, a `TEMP B-TREE`, or a correlated subquery.
- **4. Change one thing**: add one index or rewrite one clause, never both at once.
- **5. Re-measure**: compare steps, plan and result. Keep the change only if it helps and the result is identical.
- Below we follow the five steps once on a typical query: "total paid amount for customer 42".

In [4]:
%%sql
-- steps 1-3: reproduce, measure, plan (no index yet)
SELECT COUNT(*) AS paid_orders, SUM(amount) AS total FROM orders WHERE customer_id = 42 AND status = 'paid';

paid_orders | total
------------+------
5           | 1275 
(1 row)



In [5]:
q = "SELECT COUNT(*) AS paid_orders, SUM(amount) AS total FROM orders WHERE customer_id = 42 AND status = 'paid'"
base_steps, _, base_rows = steps(q)
print("baseline steps:", f"{base_steps:,}")
plan(q)

baseline steps: 15,069
SCAN orders


In [6]:
%%sql
-- step 4: change ONE thing (a composite index: equality columns first, then the summed column so it is covering)
CREATE INDEX idx_orders_cust_status_amt ON orders (customer_id, status, amount);

In [7]:
new_steps, _, new_rows = steps(q)
print("after index steps:", f"{new_steps:,}")
plan(q)
print("same result:", new_rows == base_rows, "| work reduced to about 1/%d" % round(base_steps / new_steps))

after index steps: 43
SEARCH orders USING COVERING INDEX idx_orders_cust_status_amt (customer_id=? AND status=?)
same result: True | work reduced to about 1/350


## 04. SELECT * versus the columns you need
- `SELECT *` forces SQLite to read every column of every matching row from the table, even if you only use one.
- Naming only the needed columns allows a **covering index**: the answer comes from the index alone and the table is never opened (see `07-Performance/01-Indexes-And-Query-Plans`).
- Fewer columns also means less data copied into Python, and your code does not break when someone adds a column.
- `SELECT *` inside views, `INSERT ... SELECT *` and `UNION` of two `SELECT *` are fragile: they depend on column order.

In [8]:
con.execute("CREATE INDEX idx_orders_cust_amt ON orders (customer_id, amount)")
compare({
    "SELECT *":      "SELECT * FROM orders WHERE customer_id = 42 ORDER BY amount",
    "needed columns": "SELECT order_id, amount FROM orders WHERE customer_id = 42 ORDER BY amount",
    "amount only":    "SELECT amount FROM orders WHERE customer_id = 42 ORDER BY amount",
}, check=False)   # the variants return different columns on purpose
print()
plan("SELECT * FROM orders WHERE customer_id = 42 ORDER BY amount")
print("--")
plan("SELECT amount FROM orders WHERE customer_id = 42 ORDER BY amount")

SELECT *                     steps=     192  rows=18
needed columns               steps=     101  rows=18
amount only                  steps=      83  rows=18

SEARCH orders USING INDEX idx_orders_cust_amt (customer_id=?)
--
SEARCH orders USING COVERING INDEX idx_orders_cust_amt (customer_id=?)


## 05. Functions and arithmetic on an indexed column
- An index stores the raw column values. If you wrap the column in a function (`strftime`, `substr`, `lower`) or in arithmetic (`amount * 2`), SQLite cannot look the wrapped value up in the index.
- Rewrite the condition so the **column stands alone** on one side and the computation moves to the constant side.
- Dates stored as `YYYY-MM-DD` text sort correctly, so a month becomes the range `>= first day AND < first day of next month`.
- If you truly need the function, create an expression index (`CREATE INDEX ... ON t (lower(email))`); see `07-Performance/01-Indexes-And-Query-Plans`.

In [9]:
con.execute("CREATE INDEX idx_orders_date ON orders (order_date)")
compare({
    "strftime on column": "SELECT order_id FROM orders WHERE strftime('%Y-%m', order_date) = '2024-03'",
    "substr on column":   "SELECT order_id FROM orders WHERE substr(order_date, 1, 7) = '2024-03'",
    "half-open range":    "SELECT order_id FROM orders WHERE order_date >= '2024-03-01' AND order_date < '2024-04-01'",
})
print()
plan("SELECT order_id FROM orders WHERE strftime('%Y-%m', order_date) = '2024-03'")
print("--")
plan("SELECT order_id FROM orders WHERE order_date >= '2024-03-01' AND order_date < '2024-04-01'")

strftime on column           steps=  20,848  rows=417
substr on column             steps=  20,849  rows=417
half-open range              steps=   1,683  rows=417
all results identical: True

SCAN orders USING COVERING INDEX idx_orders_date
--
SEARCH orders USING COVERING INDEX idx_orders_date (order_date>? AND order_date<?)


In [10]:
# arithmetic on the column: move the maths to the constant side
compare({
    "amount * 2 > 800": "SELECT order_id FROM orders WHERE customer_id = 42 AND amount * 2 > 800",
    "amount > 400":     "SELECT order_id FROM orders WHERE customer_id = 42 AND amount > 400",
})

amount * 2 > 800             steps=     115  rows=4
amount > 400                 steps=      33  rows=4
all results identical: True


## 06. Leading-wildcard LIKE
- `LIKE 'cust1%'` (fixed start) can become an index range. `LIKE '%example.com'` (wildcard first) cannot: the start is unknown, so every row is checked.
- For `LIKE` to use an index, the index must be `COLLATE NOCASE` (LIKE ignores case by default for ASCII). `GLOB` is case-sensitive and works with a normal index.
- "Ends with" searches: store the piece you search for in its own column (here a generated `domain` column) and index that. Equality on a small column beats a wildcard scan.
- For real word search inside long text, use FTS5 (see the SQLite-Specials track), not `LIKE '%word%'`.

In [11]:
con.execute("CREATE INDEX idx_cust_email_nc ON customers (email COLLATE NOCASE)")
compare({
    "LIKE '%@example.com'":  "SELECT customer_id FROM customers WHERE email LIKE '%@example.com'",
    "LIKE 'cust17%' (prefix)": "SELECT customer_id FROM customers WHERE email LIKE 'cust17%'",
}, check=False)   # different questions, different rows: only the work is compared
print()
plan("SELECT customer_id FROM customers WHERE email LIKE '%@example.com'")
print("--")
plan("SELECT customer_id FROM customers WHERE email LIKE 'cust17%'")

LIKE '%@example.com'         steps=   1,409  rows=100
LIKE 'cust17%' (prefix)      steps=      72  rows=11

SCAN customers USING COVERING INDEX idx_cust_email_nc
--
SEARCH customers USING COVERING INDEX idx_cust_email_nc (email>? AND email<?)


In [12]:
%%sql
-- "ends with": a generated column holds the domain, an index makes it an equality search
ALTER TABLE customers ADD COLUMN domain TEXT GENERATED ALWAYS AS (substr(email, instr(email, '@') + 1)) VIRTUAL;
CREATE INDEX idx_cust_domain ON customers (domain);
SELECT domain, COUNT(*) AS customers FROM customers GROUP BY domain ORDER BY domain;

domain      | customers
------------+----------
corp.net    | 100      
example.com | 100      
mail.org    | 100      
(3 rows)



In [13]:
compare({
    "LIKE '%@example.com'": "SELECT customer_id FROM customers WHERE email LIKE '%@example.com'",
    "domain = 'example.com'": "SELECT customer_id FROM customers WHERE domain = 'example.com'",
})
plan("SELECT customer_id FROM customers WHERE domain = 'example.com'")

LIKE '%@example.com'         steps=   1,409  rows=100
domain = 'example.com'       steps=     509  rows=100
all results identical: True
SEARCH customers USING INDEX idx_cust_domain (domain=?)


## 07. OR: IN, UNION ALL, and the duplicate trap
- `col = 1 OR col = 2` on one column is the same as `col IN (1, 2)`. SQLite plans both the same way; `IN` is shorter and easier to extend.
- `OR` across **different** columns can only use indexes if each side has one (SQLite then does a "MULTI-INDEX OR"). If one side has no index, the whole thing scans.
- Rewrite with `UNION ALL` so each branch uses its own index. But `UNION ALL` keeps duplicates: a row matching both branches appears twice. Exclude it in the second branch, or use plain `UNION` (which sorts to remove duplicates, costing extra).

In [14]:
compare({
    "OR on one column":  "SELECT order_id FROM orders WHERE customer_id = 7 OR customer_id = 8",
    "IN (7, 8)":         "SELECT order_id FROM orders WHERE customer_id IN (7, 8)",
})
plan("SELECT order_id FROM orders WHERE customer_id = 7 OR customer_id = 8")

OR on one column             steps=     172  rows=34
IN (7, 8)                    steps=     166  rows=34
all results identical: True
SEARCH orders USING COVERING INDEX idx_orders_cust_amt (customer_id=?)


In [15]:
# order_date and customer_id are indexed, ref is not: one unindexed side drags the whole OR into a scan
q_or = "SELECT order_id FROM orders WHERE customer_id = 7 OR order_date = '2024-03-02'"
q_or_bad = "SELECT order_id FROM orders WHERE customer_id = 7 OR ref = '00003'"
print("both sides indexed:")
plan(q_or)
print("\none side has no index (ref):")
plan(q_or_bad)
con.execute("CREATE INDEX idx_orders_ref ON orders (ref)")
print("\nafter indexing ref:")
plan(q_or_bad)

both sides indexed:
MULTI-INDEX OR
  INDEX 1
    SEARCH orders USING INDEX idx_orders_cust_amt (customer_id=?)
  INDEX 2
    SEARCH orders USING INDEX idx_orders_date (order_date=?)

one side has no index (ref):
SCAN orders

after indexing ref:
MULTI-INDEX OR
  INDEX 1
    SEARCH orders USING INDEX idx_orders_cust_amt (customer_id=?)
  INDEX 2
    SEARCH orders USING INDEX idx_orders_ref (ref=?)


In [16]:
# UNION ALL duplicates: rows matching both conditions come back twice unless the 2nd branch excludes them.
# Pick a date on which customer 7 really has an order, so the two conditions overlap.
d = con.execute("SELECT MIN(order_date) FROM orders WHERE customer_id = 7").fetchone()[0]
print("overlap date:", d)
compare({
    "OR (reference)": f"SELECT order_id FROM orders WHERE customer_id = 7 OR order_date = '{d}'",
    "UNION ALL (wrong)": f"""SELECT order_id FROM orders WHERE customer_id = 7
                            UNION ALL SELECT order_id FROM orders WHERE order_date = '{d}'""",
    "UNION ALL + exclusion": f"""SELECT order_id FROM orders WHERE customer_id = 7
                                UNION ALL SELECT order_id FROM orders
                                WHERE order_date = '{d}' AND customer_id IS NOT 7""",
    "UNION (dedupes by sorting)": f"""SELECT order_id FROM orders WHERE customer_id = 7
                                     UNION SELECT order_id FROM orders WHERE order_date = '{d}'""",
})

overlap date: 2024-01-01
OR (reference)               steps=   3,908  rows=431
UNION ALL (wrong)            steps=   1,747  rows=433
UNION ALL + exclusion        steps=   2,994  rows=431
UNION (dedupes by sorting)   steps=   3,476  rows=431
all results identical: False


## 08. Implicit type conversion
- SQLite columns have an **affinity** (TEXT, INTEGER, ...). When you compare a column to a value of another type, SQLite converts one side by fixed rules, and that can silently change the answer or hide the index.
- `ref` is a TEXT column holding `'00123'`. Comparing it with the number `123` converts the number to text `'123'`, which does not equal `'00123'`: **zero rows, no error**. The index is still used, so the query is fast *and wrong*.
- Wrapping the column in `CAST(ref AS INTEGER)` finds the row but cannot use the index on `ref`: the plan becomes a scan.
- In a join between an INTEGER column and a TEXT column the same conversion applies to the text column, which limits the plans the optimizer may choose (see the second cell).
- Best fix: compare with the same type as the column (`ref = '00123'`), and store IDs in a consistent type. Use bound parameters with the right Python type.

In [17]:
q = {
    "ref = 123 (number)":        "SELECT order_id FROM orders WHERE ref = 123",
    "ref = '00123' (text)":      "SELECT order_id FROM orders WHERE ref = '00123'",
    "CAST(ref AS INTEGER) = 123": "SELECT order_id FROM orders WHERE CAST(ref AS INTEGER) = 123",
}
compare(q, check=False)
print()
for label, s in q.items():
    print(label); plan(s); print()

ref = 123 (number)           steps=      10  rows=0
ref = '00123' (text)         steps=      13  rows=1
CAST(ref AS INTEGER) = 123   steps=  20,015  rows=1

ref = 123 (number)
SEARCH orders USING COVERING INDEX idx_orders_ref (ref=?)

ref = '00123' (text)
SEARCH orders USING COVERING INDEX idx_orders_ref (ref=?)

CAST(ref AS INTEGER) = 123
SCAN orders USING COVERING INDEX idx_orders_ref



In [18]:
%%sql
-- mismatched types across tables: tickets store the customer id as TEXT
CREATE TABLE tickets (ticket_id INTEGER PRIMARY KEY, customer_ref TEXT NOT NULL, topic TEXT NOT NULL);
WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 600)
INSERT INTO tickets SELECT i, CAST(1 + (i * 13) % 300 AS TEXT), CASE i % 3 WHEN 0 THEN 'billing' WHEN 1 THEN 'login' ELSE 'delivery' END FROM n;
CREATE INDEX idx_tickets_ref ON tickets (customer_ref);

In [19]:
# customers.customer_id is INTEGER, tickets.customer_ref is TEXT.
# SQLite converts the TEXT column to a number for the comparison, so the index on tickets.customer_ref cannot be used for the lookup:
# the plan has to start from tickets and probe customers instead.
plan("SELECT c.name, t.topic FROM customers c JOIN tickets t ON t.customer_ref = c.customer_id WHERE c.country = 'IN'")
print("--- same join with the type fixed by CAST on the customer side")
plan("SELECT c.name, t.topic FROM customers c JOIN tickets t ON t.customer_ref = CAST(c.customer_id AS TEXT) WHERE c.country = 'IN'")

SCAN t
SEARCH c USING INTEGER PRIMARY KEY (rowid=?)
--- same join with the type fixed by CAST on the customer side
SCAN c
SEARCH t USING INDEX idx_tickets_ref (customer_ref=?)


## 09. Correlated subquery to join or window
- A **correlated subquery** refers to the outer row, so in principle it runs once per outer row. 5000 outer rows can mean 5000 inner scans.
- Two cures: give the inner query an index on the correlation column, or compute the inner value **once** with `GROUP BY` plus a join, or a window function (see `03-Window-Functions/01-Window-Functions`).
- Task: orders whose amount is above that customer's own average. First without any index on `customer_id`: the correlated form re-scans `orders` for every outer row (the step count is shown in units of 1000 because it is so large).
- Then add the index and measure again: the correlated query improves enormously, but computing each average **once** (join or window) is still cheaper.

In [20]:
con.execute("DROP INDEX idx_orders_cust_amt")
con.execute("DROP INDEX idx_orders_cust_status_amt")
correlated = """SELECT o.order_id FROM orders o
                WHERE o.amount > (SELECT AVG(o2.amount) FROM orders o2 WHERE o2.customer_id = o.customer_id)"""
joined = """SELECT o.order_id FROM orders o
            JOIN (SELECT customer_id, AVG(amount) AS avg_amt FROM orders GROUP BY customer_id) a
              ON a.customer_id = o.customer_id
            WHERE o.amount > a.avg_amt"""
windowed = """SELECT order_id FROM (
                SELECT order_id, amount, AVG(amount) OVER (PARTITION BY customer_id) AS avg_amt FROM orders
              ) WHERE amount > avg_amt"""
compare({"correlated, no index": correlated, "join to GROUP BY": joined, "window AVG OVER": windowed}, every=1000)   # counted in units of 1000 steps
print()
plan(correlated)

correlated, no index         steps=100,288,000  rows=2495
join to GROUP BY             steps= 146,000  rows=2495
window AVG OVER              steps= 232,000  rows=2495
all results identical: True

SCAN o
CORRELATED SCALAR SUBQUERY 1
  SCAN o2


In [21]:
# the inner query becomes cheap once the correlation column is indexed
con.execute("CREATE INDEX idx_orders_cust_amt ON orders (customer_id, amount)")
compare({"correlated, with index": correlated, "join to GROUP BY": joined, "window AVG OVER": windowed})
print()
plan(correlated)

correlated, with index       steps= 441,053  rows=2495
join to GROUP BY             steps=  56,482  rows=2495
window AVG OVER              steps= 192,589  rows=2495
all results identical: True

SCAN o USING COVERING INDEX idx_orders_cust_amt
CORRELATED SCALAR SUBQUERY 1
  SEARCH o2 USING COVERING INDEX idx_orders_cust_amt (customer_id=?)


## 10. NOT IN versus NOT EXISTS
- "Customers with no orders" is an anti-join. Three ways: `NOT IN`, `NOT EXISTS`, `LEFT JOIN ... WHERE right.key IS NULL`.
- **The trap:** `x NOT IN (list)` is UNKNOWN (not true) as soon as the list contains a NULL. Our guest order has `customer_id NULL`, so `NOT IN` returns **no rows** even though 19 customers never ordered.
- `NOT EXISTS` and `LEFT JOIN ... IS NULL` compare row by row and ignore the NULL, so they give the right answer. On this data `NOT EXISTS` also needs the fewest steps because it probes the index once per customer.
- If you do use `NOT IN`, filter the NULLs out of the subquery: `WHERE customer_id IS NOT NULL`.

In [22]:
compare({"NOT IN (has a NULL)": "SELECT customer_id FROM customers WHERE customer_id NOT IN (SELECT customer_id FROM orders)"}, check=False)
print("-- the correct forms:")
compare({
    "NOT IN + IS NOT NULL":  "SELECT customer_id FROM customers WHERE customer_id NOT IN (SELECT customer_id FROM orders WHERE customer_id IS NOT NULL)",
    "NOT EXISTS":            "SELECT c.customer_id FROM customers c WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)",
    "LEFT JOIN IS NULL":     "SELECT c.customer_id FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id WHERE o.order_id IS NULL",
})

NOT IN (has a NULL)          steps=   1,889  rows=0
-- the correct forms:
NOT IN + IS NOT NULL         steps=  23,355  rows=19
NOT EXISTS                   steps=   3,311  rows=19
LEFT JOIN IS NULL            steps=  26,682  rows=19
all results identical: True


In [23]:
%%sql
-- the three correct forms agree; the first form returned nothing because of the NULL
SELECT c.customer_id, c.name FROM customers c
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
ORDER BY c.customer_id LIMIT 5;

customer_id | name        
------------+-------------
282         | Customer 282
283         | Customer 283
284         | Customer 284
285         | Customer 285
286         | Customer 286
(5 rows)



In [24]:
plan("SELECT c.customer_id FROM customers c WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)")

SCAN c USING COVERING INDEX idx_cust_domain
CORRELATED SCALAR SUBQUERY 1
  SEARCH o USING COVERING INDEX idx_orders_cust_amt (customer_id=?)


## 11. DISTINCT as a band-aid
- If a query returns duplicates and you add `DISTINCT` to hide them, the real problem is usually a **join that multiplies rows** (one customer, many orders). The database produces all the extra rows and then throws the repeats away.
- Here the join visits all 5000 order rows to find 281 distinct customers. If you only need "customers who have at least one order", ask exactly that with `EXISTS`. No duplicates are ever made.
- `DISTINCT` is right when duplicates are real in the data and you want unique values (e.g. the distinct countries); it is wrong as a patch for a bad join.

In [25]:
compare({
    "JOIN + DISTINCT":  "SELECT DISTINCT c.customer_id, c.name FROM customers c JOIN orders o ON o.customer_id = c.customer_id",
    "EXISTS":           "SELECT c.customer_id, c.name FROM customers c WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)",
})
print("rows before DISTINCT:", con.execute("SELECT COUNT(*) FROM customers c JOIN orders o ON o.customer_id = c.customer_id").fetchone()[0])
print()
plan("SELECT DISTINCT c.customer_id, c.name FROM customers c JOIN orders o ON o.customer_id = c.customer_id")
print("--")
plan("SELECT c.customer_id, c.name FROM customers c WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)")

JOIN + DISTINCT              steps=  31,469  rows=281
EXISTS                       steps=   4,114  rows=281
all results identical: True
rows before DISTINCT: 5000

SCAN c
SEARCH o USING COVERING INDEX idx_orders_cust_amt (customer_id=?)
--
SCAN c
CORRELATED SCALAR SUBQUERY 1
  SEARCH o USING COVERING INDEX idx_orders_cust_amt (customer_id=?)


## 12. COUNT(*) versus EXISTS
- `SELECT COUNT(*) ... > 0` counts **every** matching row just to learn "is there at least one?". `EXISTS` stops at the first match.
- The gap grows with the number of matches: the more matching rows, the more work `COUNT(*)` wastes.
- Use `COUNT(*)` only when you really need the number.
- Plain `SELECT COUNT(*) FROM big_table` has no shortcut in SQLite: it walks the smallest index or the table. Keep your own counter if you need it constantly.

In [26]:
con.execute("CREATE INDEX idx_orders_status ON orders (status)")
compare({
    "COUNT(*) > 0":  "SELECT COUNT(*) > 0 AS has_returned FROM orders WHERE status = 'returned'",
    "EXISTS":        "SELECT EXISTS (SELECT 1 FROM orders WHERE status = 'returned') AS has_returned",
})
print()
plan("SELECT COUNT(*) FROM orders")

COUNT(*) > 0                 steps=   3,765  rows=1
EXISTS                       steps=      18  rows=1
all results identical: True

SCAN orders USING COVERING INDEX idx_orders_cust_amt


## 13. Pagination: OFFSET versus keyset
- `LIMIT 20 OFFSET 4000` still walks past 4000 rows and throws them away. Page 1 is cheap, page 200 is not.
- **Keyset pagination** remembers the last row of the previous page and asks for rows *after* it: `WHERE (order_date, order_id) > (?, ?) ORDER BY order_date, order_id LIMIT 20`. With an index on those columns SQLite jumps straight there.
- The sort key must be **unique** (add the primary key as a tiebreaker) or rows are skipped or repeated.
- Keyset still starts from the last date seen, so it reads the few rows tied on that date; the gain is skipping all the earlier pages.
- Trade-off: keyset gives "next page" / "previous page", not "jump to page 57".

In [27]:
con.execute("CREATE INDEX idx_orders_date_id ON orders (order_date, order_id)")
page_size, page_no = 20, 200
offset = (page_no - 1) * page_size
last = con.execute(
    "SELECT order_date, order_id FROM orders ORDER BY order_date, order_id LIMIT 1 OFFSET ?", (offset - 1,)
).fetchone()
print("last row of the previous page:", last)
off_q = f"SELECT order_id, order_date FROM orders ORDER BY order_date, order_id LIMIT {page_size} OFFSET {offset}"
key_q = f"SELECT order_id, order_date FROM orders WHERE (order_date, order_id) > ('{last[0]}', {last[1]}) ORDER BY order_date, order_id LIMIT {page_size}"
compare({"OFFSET 3980": off_q, "keyset": key_q})
print()
plan(key_q)

last row of the previous page: ('2024-10-08', 2721)
OFFSET 3980                  steps=   8,090  rows=20
keyset                       steps=   2,300  rows=20
all results identical: True

SEARCH orders USING COVERING INDEX idx_orders_date_id (order_date>?)


In [28]:
# page 1 for comparison: both are cheap there, the cost of OFFSET grows with the page number
compare({"OFFSET 0":  f"SELECT order_id FROM orders ORDER BY order_date, order_id LIMIT {page_size} OFFSET 0",
         "OFFSET 2000": f"SELECT order_id FROM orders ORDER BY order_date, order_id LIMIT {page_size} OFFSET 2000"}, check=False)

OFFSET 0                     steps=     110  rows=20
OFFSET 2000                  steps=   4,110  rows=20


## 14. N+1 queries versus one join
- The N+1 pattern: one query fetches N rows, then your code runs **one more query per row** (N extra queries). It is the most common slowdown in application code.
- SQLite runs inside your process, so each query is cheap, but every query still costs parsing, planning and a Python call. Against a client/server database each one is also a network round trip.
- The fix is one query that does the joining and grouping, so the database loops, not Python.
- We count queries with a tiny wrapper object around the connection. The count is exact and the same every run.

In [29]:
class Counted:
    """Counts how many queries are sent through it."""
    def __init__(self, connection):
        self.con, self.queries = connection, 0
    def execute(self, sql, params=()):
        self.queries += 1
        return self.con.execute(sql, params)

# N+1: one query for the customers, then one query per customer
db = Counted(con)
customers_in = db.execute("SELECT customer_id, name FROM customers WHERE country = 'IN' ORDER BY customer_id").fetchall()
n_plus_1 = []
for cid, name in customers_in:
    total = db.execute("SELECT COALESCE(SUM(amount), 0) FROM orders WHERE customer_id = ?", (cid,)).fetchone()[0]
    n_plus_1.append((cid, name, total))
print("N+1 approach: customers =", len(customers_in), "| queries sent =", db.queries)

# one query: join and group in the database
db = Counted(con)
one_query = db.execute("""
    SELECT c.customer_id, c.name, COALESCE(SUM(o.amount), 0)
    FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id
    WHERE c.country = 'IN'
    GROUP BY c.customer_id, c.name ORDER BY c.customer_id""").fetchall()
print("single join  : queries sent =", db.queries)
print("same result  :", one_query == n_plus_1)

N+1 approach: customers = 75 | queries sent = 76
single join  : queries sent = 1
same result  : True


## 15. Batching IN lists
- You often have a Python list of ids and need their rows. Looping one query per id is N+1 again.
- Build one `IN (?, ?, ?, ...)` with one placeholder per id and pass the list as parameters. Never paste values into the SQL text (injection risk and no plan reuse).
- Every build has a limit on bound parameters (`SQLITE_LIMIT_VARIABLE_NUMBER`); for long lists send chunks of a few hundred.
- Another trick: pass the whole list as **one** JSON parameter and unpack it with `json_each`. One placeholder, any length.

In [30]:
import sqlite3
ids = list(range(1, 901, 3))   # 300 customer ids
print("ids:", len(ids), "| parameter limit is large enough:", con.getlimit(sqlite3.SQLITE_LIMIT_VARIABLE_NUMBER) >= 300)

def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

# a) one query per id
db = Counted(con); a = []
for i in ids:
    a += db.execute("SELECT customer_id, name FROM customers WHERE customer_id = ?", (i,)).fetchall()
print("per-id loop   : queries =", db.queries, "| rows =", len(a))

# b) chunks of 100 placeholders
db = Counted(con); b = []
for part in chunks(ids, 100):
    marks = ",".join("?" * len(part))
    b += db.execute(f"SELECT customer_id, name FROM customers WHERE customer_id IN ({marks})", part).fetchall()
print("chunks of 100 : queries =", db.queries, "| rows =", len(b))

# c) one JSON parameter
import json
db = Counted(con)
c = db.execute("SELECT customer_id, name FROM customers WHERE customer_id IN (SELECT value FROM json_each(?))",
               (json.dumps(ids),)).fetchall()
print("json_each     : queries =", db.queries, "| rows =", len(c))
print("same rows     :", sorted(a) == sorted(b) == sorted(c))
plan("SELECT customer_id, name FROM customers WHERE customer_id IN (SELECT value FROM json_each(?))", (json.dumps(ids),))

ids: 300 | parameter limit is large enough: True
per-id loop   : queries = 300 | rows = 100
chunks of 100 : queries = 3 | rows = 100
json_each     : queries = 1 | rows = 100
same rows     : True
SEARCH customers USING INTEGER PRIMARY KEY (rowid=?)
LIST SUBQUERY 1
  SCAN json_each VIRTUAL TABLE INDEX 1:


## 16. Avoiding the sort with an index
- `ORDER BY` on a column with no suitable index makes SQLite collect all matching rows and sort them: the plan says `USE TEMP B-TREE FOR ORDER BY`.
- If an index already stores rows in the wanted order, SQLite just reads the index in order and the sort disappears. With `LIMIT 10` it can stop after 10 rows.
- The index must match the `ORDER BY` columns **in order** and after any equality columns in `WHERE` (see "leftmost prefix" in `07-Performance/01-Indexes-And-Query-Plans`).
- `ORDER BY ... DESC` is fine: an index can be read backwards. Mixing ASC and DESC across columns is not (unless the index says so).

In [31]:
compare({
    "top 10 by amount (no index)":       "SELECT order_id, amount FROM orders ORDER BY amount DESC, order_id LIMIT 10",
    "top 10 by date (index on date,id)": "SELECT order_id, order_date FROM orders ORDER BY order_date, order_id LIMIT 10",
}, check=False)
plan("SELECT order_id, amount FROM orders ORDER BY amount DESC, order_id LIMIT 10")
print("--")
plan("SELECT order_id, order_date FROM orders ORDER BY order_date, order_id LIMIT 10")
print("--")
plan("SELECT order_id FROM orders ORDER BY order_date DESC, order_id DESC LIMIT 10")

top 10 by amount (no index)  steps=  35,296  rows=10
top 10 by date (index on date,id) steps=      57  rows=10
SCAN orders USING COVERING INDEX idx_orders_cust_amt
USE TEMP B-TREE FOR ORDER BY
--
SCAN orders USING COVERING INDEX idx_orders_date_id
--
SCAN orders USING COVERING INDEX idx_orders_date_id


In [32]:
# equality column first, then the ORDER BY column: still no sort
plan("SELECT order_id, amount FROM orders WHERE customer_id = 7 ORDER BY amount")
print("-- ORDER BY a column that is not in the index: sort comes back")
plan("SELECT order_id FROM orders WHERE customer_id = 7 ORDER BY order_date")

SEARCH orders USING COVERING INDEX idx_orders_cust_amt (customer_id=?)
-- ORDER BY a column that is not in the index: sort comes back
SEARCH orders USING INDEX idx_orders_cust_amt (customer_id=?)
USE TEMP B-TREE FOR ORDER BY


## 17. Covering queries for aggregates
- `07-Performance/01-Indexes-And-Query-Plans` explains covering indexes. Here is the optimization angle: a `GROUP BY` that needs only indexed columns is answered **from the index alone**, already in group order, so there is no table access and no sort.
- The moment the query needs one column that is not in the index, SQLite must visit the table for every row.
- Remedy: add the missing column to the index (a covering index). The price is more disk space and slower writes, so do it only for queries that matter.

In [33]:
covered   = "SELECT customer_id, SUM(amount) AS total FROM orders GROUP BY customer_id"
uncovered = "SELECT customer_id, SUM(amount) AS total FROM orders WHERE status = 'paid' GROUP BY customer_id"
# the two queries return different numbers of rows, so look at the plan text, not at the step counts, to compare them
compare({"GROUP BY, covered": covered, "WHERE status + GROUP BY": uncovered}, check=False)
plan(covered); print("--"); plan(uncovered)

GROUP BY, covered            steps=  43,964  rows=282
WHERE status + GROUP BY      steps=  23,978  rows=282
SCAN orders USING COVERING INDEX idx_orders_cust_amt
--
SEARCH orders USING INDEX idx_orders_status (status=?)
USE TEMP B-TREE FOR GROUP BY


In [34]:
con.execute("CREATE INDEX idx_orders_status_cust_amt ON orders (status, customer_id, amount)")
compare({"WHERE status + GROUP BY, new index": uncovered}, check=False)
plan(uncovered)

WHERE status + GROUP BY, new index steps=  15,219  rows=282
SEARCH orders USING COVERING INDEX idx_orders_status_cust_amt (status=?)


## 18. CTEs and subqueries: flattening and materialization
- A CTE (`WITH name AS (...)`) is only a name. SQLite usually **flattens** it into the outer query, so filters on the outside reach the inside and indexes still work.
- A CTE that is used **more than once** is computed once and stored (the plan says `MATERIALIZE`). Good when it is expensive; wasteful when you only needed a few rows of it.
- You can force the choice: `AS MATERIALIZED` (compute once, store) or `AS NOT MATERIALIZED` (inline it every time).
- A subquery with `LIMIT`, `DISTINCT` or an aggregate cannot always be flattened; SQLite may still push a simple filter on a `GROUP BY` column inside it.

In [35]:
cte_once = """WITH recent AS (SELECT order_id, customer_id, amount FROM orders WHERE order_date >= '2024-12-01')
              SELECT order_id, amount FROM recent WHERE customer_id = 7 ORDER BY order_id"""
print("CTE used once (flattened into the outer query):"); plan(cte_once)
cte_twice = """WITH per_cust AS (SELECT customer_id, SUM(amount) AS total FROM orders GROUP BY customer_id)
               SELECT a.customer_id, a.total, b.total AS next_total
               FROM per_cust a JOIN per_cust b ON b.customer_id = a.customer_id + 1
               WHERE a.customer_id <= 5 ORDER BY a.customer_id"""
print("\nCTE used twice (computed once and stored):"); plan(cte_twice)

CTE used once (flattened into the outer query):
SEARCH orders USING INDEX idx_orders_cust_amt (customer_id=?)
USE TEMP B-TREE FOR ORDER BY

CTE used twice (computed once and stored):
MATERIALIZE per_cust
  SCAN orders USING COVERING INDEX idx_orders_cust_amt
SCAN a
BLOOM FILTER ON b (customer_id=?)
SEARCH b USING AUTOMATIC COVERING INDEX (customer_id=?)
USE TEMP B-TREE FOR ORDER BY


In [36]:
base = "SELECT customer_id, SUM(amount) AS total FROM orders GROUP BY customer_id"
q_mat   = f"WITH s AS MATERIALIZED ({base}) SELECT * FROM s WHERE customer_id = 7"
q_inl   = f"WITH s AS NOT MATERIALIZED ({base}) SELECT * FROM s WHERE customer_id = 7"
compare({"AS MATERIALIZED": q_mat, "AS NOT MATERIALIZED": q_inl})
print("\nMATERIALIZED:"); plan(q_mat)
print("\nNOT MATERIALIZED:"); plan(q_inl)

AS MATERIALIZED              steps=  45,386  rows=1
AS NOT MATERIALIZED          steps=     201  rows=1
all results identical: True

MATERIALIZED:
MATERIALIZE s
  SCAN orders USING COVERING INDEX idx_orders_cust_amt
SCAN s

NOT MATERIALIZED:
CO-ROUTINE s
  SEARCH orders USING COVERING INDEX idx_orders_cust_amt (customer_id=?)
SCAN s


## 19. ANALYZE changes plans
- SQLite chooses plans from guesses about your data. `ANALYZE` measures real selectivity and stores it in `sqlite_stat1`; the planner then makes better choices (`07-Performance/01-Indexes-And-Query-Plans` shows the table).
- Below, the table has 95% `'view'` rows and 5% `'purchase'` rows. Before `ANALYZE` the planner has no idea an equality on `kind` is a weak filter, so it starts the join from the big table. After it, the plan flips.
- Run `ANALYZE` (or `PRAGMA optimize`, see `07-Performance/03-Pragmas-Storage-And-Maintenance`) after bulk loads or big data changes.

In [37]:
%%sql
CREATE TABLE group_labels (grp INTEGER PRIMARY KEY, label TEXT NOT NULL);
CREATE TABLE events (event_id INTEGER PRIMARY KEY, kind TEXT NOT NULL, grp INTEGER NOT NULL, note TEXT NOT NULL);
WITH RECURSIVE n(i) AS (SELECT 0 UNION ALL SELECT i + 1 FROM n WHERE i < 39)
INSERT INTO group_labels SELECT i, 'g' || i FROM n;
WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 4000)
INSERT INTO events SELECT i, CASE WHEN i % 20 = 0 THEN 'purchase' ELSE 'view' END, i % 40, 'x' FROM n;
CREATE INDEX idx_events_kind_grp ON events (kind, grp);
SELECT kind, COUNT(*) AS rows_ FROM events GROUP BY kind ORDER BY kind;

kind     | rows_
---------+------
purchase | 200  
view     | 3800 
(2 rows)



In [38]:
join_q = "SELECT e.event_id FROM events e JOIN group_labels g ON g.grp = e.grp WHERE e.kind = 'view' AND g.label = 'g3'"
before_steps, before_rows, _ = steps(join_q)
print("BEFORE ANALYZE: steps =", f"{before_steps:,}", "| rows =", before_rows); plan(join_q)
con.execute("ANALYZE")
after_steps, after_rows, _ = steps(join_q)
print("\nAFTER ANALYZE: steps =", f"{after_steps:,}", "| rows =", after_rows); plan(join_q)

BEFORE ANALYZE: steps = 23,012 | rows = 100
SEARCH e USING COVERING INDEX idx_events_kind_grp (kind=?)
SEARCH g USING INTEGER PRIMARY KEY (rowid=?)

AFTER ANALYZE: steps = 536 | rows = 100
SCAN g
SEARCH e USING COVERING INDEX idx_events_kind_grp (kind=? AND grp=?)


In [39]:
%%sql
SELECT tbl, idx, stat FROM sqlite_stat1 WHERE tbl IN ('events', 'group_labels') ORDER BY tbl, idx;

tbl          | idx                 | stat         
-------------+---------------------+--------------
events       | idx_events_kind_grp | 4000 2000 100
group_labels | NULL                | 40           
(2 rows)



## 20. Semi-joins: IN, EXISTS and JOIN
- A **semi-join** keeps rows from table A that have a match in B, without multiplying A's rows. Question shape: "customers who have at least one big recent order".
- Three spellings: `IN (subquery)`, `EXISTS (correlated subquery)`, and a join followed by `DISTINCT`. They return the same rows.
- On this data all three finish within a small factor of each other (see the step counts). The join + `DISTINCT` form looks fine here only because few orders match; with many matches per customer it produces many duplicate rows first (section 11). `EXISTS` stops at the first match and states the intent.
- SQLite's plan text calls a correlated `EXISTS` a `CORRELATED SCALAR SUBQUERY`; do not let the name worry you, it is a search per customer using the index.
- Pick one style and use it consistently. Measure when it matters.

In [40]:
semi_in = """SELECT customer_id, name FROM customers
             WHERE customer_id IN (SELECT customer_id FROM orders WHERE status = 'returned' AND amount > 450)"""
semi_ex = """SELECT c.customer_id, c.name FROM customers c
             WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id AND o.status = 'returned' AND o.amount > 450)"""
semi_jd = """SELECT DISTINCT c.customer_id, c.name FROM customers c
             JOIN orders o ON o.customer_id = c.customer_id WHERE o.status = 'returned' AND o.amount > 450"""
compare({"IN (subquery)": semi_in, "EXISTS": semi_ex, "JOIN + DISTINCT": semi_jd})
for label, s in [("IN", semi_in), ("EXISTS", semi_ex), ("JOIN + DISTINCT", semi_jd)]:
    print("\n" + label); plan(s)

IN (subquery)                steps=   6,293  rows=127
EXISTS                       steps=   4,261  rows=127
JOIN + DISTINCT              steps=   2,711  rows=127
all results identical: True

IN
SEARCH customers USING INTEGER PRIMARY KEY (rowid=?)
LIST SUBQUERY 1
  SEARCH orders USING COVERING INDEX idx_orders_status_cust_amt (status=?)

EXISTS
SCAN c
CORRELATED SCALAR SUBQUERY 1
  SEARCH o USING COVERING INDEX idx_orders_status_cust_amt (status=? AND customer_id=? AND amount>?)

JOIN + DISTINCT
SCAN c
SEARCH o USING COVERING INDEX idx_orders_status_cust_amt (status=? AND customer_id=? AND amount>?)


## 21. Aggregate early: push work below the join
- If you join first and group afterwards, every order row travels through the join. If you group the big table **first** (one row per customer) and join the small result, the join handles far fewer rows.
- Similarly, filter rows in `WHERE` (before grouping) rather than `HAVING` (after grouping) whenever the condition is about a plain column, not an aggregate. `HAVING status = 'paid'` is worse than slow: `status` is not grouped, so SQLite tests an arbitrary row of each group and returns different totals (the second cell shows the totals differ).
- The result must stay identical: this is safe when the join key is the same as the grouping key and the joined table adds one row per key (here: one customer per `customer_id`).

In [41]:
late = """SELECT c.country, SUM(o.amount) AS revenue
          FROM orders o JOIN customers c ON c.customer_id = o.customer_id
          GROUP BY c.country ORDER BY c.country"""
early = """SELECT c.country, SUM(t.total) AS revenue
           FROM (SELECT customer_id, SUM(amount) AS total FROM orders GROUP BY customer_id) t
           JOIN customers c ON c.customer_id = t.customer_id
           GROUP BY c.country ORDER BY c.country"""
compare({"join, then group": late, "group, then join": early})

join, then group             steps=  75,976  rows=4
group, then join             steps=  48,820  rows=4
all results identical: True


In [42]:
having_q = "SELECT customer_id, SUM(amount) AS total FROM orders GROUP BY customer_id HAVING status = 'paid'"
# HAVING on a non-aggregate column is allowed by SQLite but is evaluated after grouping, on an arbitrary row of each group: a wrong-answer trap
where_q = "SELECT customer_id, SUM(amount) AS total FROM orders WHERE status = 'paid' GROUP BY customer_id"
compare({"HAVING status (trap)": having_q, "WHERE status": where_q}, check=False)
a, b = dict(con.execute(having_q).fetchall()), dict(con.execute(where_q).fetchall())
common = a.keys() & b.keys()
print("customers returned by both:", len(common), "| of those with the same total:", sum(a[k] == b[k] for k in common))

HAVING status (trap)         steps=  48,892  rows=69
WHERE status                 steps=  15,219  rows=282
customers returned by both: 69 | of those with the same total: 1


In [43]:
# a legitimate HAVING: the condition depends on the aggregate
compare({"HAVING SUM > 4000": "SELECT customer_id, SUM(amount) AS total FROM orders GROUP BY customer_id HAVING SUM(amount) > 4000"}, check=False)

HAVING SUM > 4000            steps=  44,196  rows=265


## 22. Before/after checklist
- **Before you change anything:** have the exact query, realistic data, a baseline step count, the row count and the plan.
- **Read the plan:** big `SCAN`? `USE TEMP B-TREE`? `CORRELATED SCALAR SUBQUERY` over a big table?
- **Rewrite checklist:** name columns instead of `*`; keep indexed columns bare; avoid leading `%`; `IN` for same-column OR; `NOT EXISTS` instead of `NOT IN`; `EXISTS` instead of `COUNT(*) > 0` or `DISTINCT`; keyset instead of big OFFSET; one join instead of N+1; aggregate before joining.
- **After:** same rows? fewer steps? plan simpler? If not, undo it.
- **Cost of indexes:** each one slows writes and takes space. The listing below is every index created in this notebook; in a real project each one should be able to explain which query it serves.

In [44]:
%%sql
SELECT tbl_name, name, CASE WHEN sql LIKE '%COLLATE NOCASE%' THEN 'nocase' ELSE '' END AS note
FROM sqlite_master WHERE type = 'index' AND name NOT LIKE 'sqlite_autoindex%'
ORDER BY tbl_name, name;

tbl_name  | name                       | note  
----------+----------------------------+-------
customers | idx_cust_domain            |       
customers | idx_cust_email_nc          | nocase
events    | idx_events_kind_grp        |       
orders    | idx_orders_cust_amt        |       
orders    | idx_orders_date            |       
orders    | idx_orders_date_id         |       
orders    | idx_orders_ref             |       
orders    | idx_orders_status          |       
orders    | idx_orders_status_cust_amt |       
tickets   | idx_tickets_ref            |       
(10 rows)

